In [ ]:
'''
Runs all Uniform baseline experiments for AlphaFold protein residue disorder rate (finite-population).
'''
import torch
import numpy as np
from scipy.stats import norm
from tqdm.autonotebook import tqdm
import os, csv

# set a device + specify number of seeds, uniform-mixing & CI target coverage + spread of budgets
device = "cuda" if torch.cuda.is_available() else "cpu"
N_SEEDS, ALPHA = 1000, 0.05
BUDGET_PROPS = np.round(np.linspace(0.0, 0.25, 11)[1:], decimals=3)

# filename for this simulation
fname = f"alphafold_uniform.csv"

# dataframe to store all of our results
columns = ["dataset", "budget", "tau", "method", "seed", "thetahat", "coverage", "width", "n_labels"]
if fname not in os.listdir("../logs"):
    with open(f"../logs/{fname}", "wt") as file:
        file.write(",".join(columns) + "\n")

In [ ]:
#### LOAD IN OUR DATA

# for alphafold + uniform, just need y_test
ys = torch.load("../datasets/alphafold/y_test.pt").to(device) # (10802,)

# get our finite-population size
N = ys.shape[0]

In [ ]:
#### UNIFORM/Bernoulli SAMPLING

# iterate through our budgets
for budget_idx, budget in enumerate(BUDGET_PROPS):
    
    # what is our (expected) budget
    n = int(budget * N)
    
    #### initialize data structures
    
    # set a seed for reproducibility
    torch.random.manual_seed(858 + budget_idx)

    # how many labels have we used up already
    obs_counts = torch.zeros(size=(N_SEEDS, 1), dtype=torch.float32, device=device)

    # recurring sum (WITHOUT NORMALIZATION) of thetahat (need div by N) and sigmahat^2 (need div by N^2)
    thetahats_unnormalized = torch.zeros(size=(N_SEEDS, 1), device=device, dtype=torch.float32)
    sigmahats_sq_unnormalized = torch.zeros(size=(N_SEEDS, 1), device=device, dtype=torch.float32)
    
    #### uniform/bernoulli sampling (need loop thru full dataset, no budget stabilization.)
    for t in tqdm(range(N), desc=f"Budget {budget_idx + 1} of {len(BUDGET_PROPS)}"):
    
        # under uniform sampling, pi_t = n / N always.
        pi_t = n / N

        # are we going to query this data point ir not?
        xi_t = torch.bernoulli(torch.full(size=(N_SEEDS, 1), fill_value=pi_t, device=device)) # (N_SEEDS, 1)

        #### PART 2. UPDATING SEQUENTIAL ACTIVE INFERENCE-TYPE ESTIMATOR + BOOKKEEPING

        # update our means and variances
        thetahats_unnormalized += (xi_t / pi_t) * ys[t]
        sigmahats_sq_unnormalized += (xi_t * (1.0 - pi_t) / (pi_t ** 2) ) * (ys[t] ** 2)

        # did we exhaust budget? also, did we hit a big-residual datapoint?
        obs_counts += xi_t

    #### COMPUTING SEQUENTIAL ACTIVE INFERENCE-TYPE CONFIDENCE INTERVAL AND COVERAGE
    
    # dummy tensor for brevity
    ZERO = torch.tensor(0.0, device=device)

    # get our mean estimate + variance estimate
    thetahats = thetahats_unnormalized / N
    sigmahats = torch.sqrt(torch.maximum(sigmahats_sq_unnormalized, ZERO) / N)
    sigmahats_bar = sigmahats / torch.sqrt(torch.tensor(N, device=device))

    # get our z-score + the lower/upper limits of our interval
    z_score = norm.ppf(1 - (ALPHA / 2))
    lbs = thetahats - z_score * sigmahats_bar
    ubs = thetahats + z_score * sigmahats_bar
    widths = (ubs - lbs).cpu().numpy()
    coverages = ((lbs <= ys.mean(dim=-1, keepdim=True)) & (ys.mean(dim=-1, keepdim=True) <= ubs)).cpu().numpy()
    
    # for file writing purposes, convert thetahats + obs_counts to numpy
    thetahats = thetahats.cpu().numpy()
    obs_counts = obs_counts.cpu().numpy()
    
    # format results + directly write to file
    results = [
        ["alphafold", budget, 0.0, "uniform", seed, 
         thetahats[seed][0], coverages[seed][0], widths[seed][0], obs_counts[seed][0]
        ] for seed in range(N_SEEDS)]
    with open(f"../logs/{fname}", "a", newline="") as file:
        writer = csv.writer(file)
        writer.writerows(results)